# 20.11 字都認對，為什麼整頁仍可能讀錯？


購物單第一行是「牛奶」，第二行是「麵包」。如果助理先讀麵包，字沒有少，順序卻變了。完整轉寫除了認字，還需要保留題目指定的行與版面次序。

[11.16](https://birdhackor.github.io/tiny-perceptron-vlm/11.16.html)已分清字元與字序。這裡先用橫排兩行，約定由上往下、每行由左到右。直排、多欄或幾塊分散招牌可能有別的讀法；這些規則應在資料標註中說清楚，不交給評分者看完答案後臨時決定。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_reading_order.svg")))

In [ ]:
from collections import Counter

reference = "牛奶\n麵包"
predicted = "麵包\n牛奶"
print("字元種類與數量相同", Counter(reference) == Counter(predicted))
print("完整字串相同", reference == predicted)
print("參考行序", reference.splitlines())
print("預測行序", predicted.splitlines())

`Counter`只數每種字元出現幾次，所以第一行True；它像把兩份購物單都倒進同一個字袋，丟掉了排列。完整字串比較是False。`\n`表示換行，`splitlines()`把文字按行分開，後兩項清楚列出上下交換。

移除全部空白可以方便某些文字內容比較，卻也移除了換行結構。這時就不能只靠「整理後一樣」宣稱讀序完整。對長頁面，最好保存文字區域、各區域內容與閱讀順序；只有一串攤平後的答案，未必足以評完整佈局。

圖片裁切像拿放大鏡：容易看清小字，但如果忘了原位置，就不知道這行原本在頁首還是頁尾。訓練文字區域與訓練整頁版面是不同用途。完整成品的能力卡也會把這兩項分開，不因有OCR分數就自動打勾所有多欄文件。

練習把`predicted`改成「牛奶麵包」。哪些字仍正確，哪個結構消失？再想一張左右兩欄的單據，應先沿欄讀，還是每一行跨兩欄讀？答案需要版面規約。下一節再把「先後相接」的觀念帶到語音與聊天。
